# 🏥 Hospital Readmission Rate Analysis — CMS FY2026
## Notebook 1: Exploratory Data Analysis & Cleaning
**Author:** Loknadh V K S Kona | [GitHub](https://github.com/KrishnaSai315) | [LinkedIn](https://linkedin.com/in/lvkrishna3)  
**Data Source:** CMS Hospital Readmissions Reduction Program (HRRP) FY2026 — data.cms.gov  
**Stack:** Python · Pandas · NumPy

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded ')

Libraries loaded 


## 1. Load Raw Data

In [7]:
# ── File paths — update if your files are in a different location
HRRP_PATH = r'D:\AG\Health Care Project\FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv'
HOSP_PATH = r'D:\AG\Health Care Project\Hospital_General_Information.csv'

# Pandas handles all CSV quoting natively — no pipe workaround needed here
df_raw   = pd.read_csv(HRRP_PATH)
df_hosp  = pd.read_csv(HOSP_PATH)

print(f'HRRP dataset   : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Hospital info  : {df_hosp.shape[0]:,} rows × {df_hosp.shape[1]} columns')

HRRP dataset   : 18,330 rows × 12 columns
Hospital info  : 5,432 rows × 38 columns


## 2. Initial Inspection

In [10]:
# Column names — confirmed from actual FY2026 file
print('HRRP columns:')
for i, col in enumerate(df_raw.columns, 1):
    print(f'  {i:2}. {col}')

HRRP columns:
   1. Facility Name
   2. Facility ID
   3. State
   4. Measure Name
   5. Number of Discharges
   6. Footnote
   7. Excess Readmission Ratio
   8. Predicted Readmission Rate
   9. Expected Readmission Rate
  10. Number of Readmissions
  11. Start Date
  12. End Date


In [12]:
df_raw.head(3)

,Facility Name,Facility ID,State,Measure Name,Number of Discharges,Footnote,Excess Readmission Ratio,Predicted Readmission Rate,Expected Readmission Rate,Number of Readmissions,Start Date,End Date
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,Too Few to Report,07/01/2021,06/30/2024
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137.0000,NaN,0.9531,10.3960,10.9078,13,07/01/2021,06/30/2024
2,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-AMI-HRRP,273.0000,NaN,0.9370,13.2998,14.1948,33,07/01/2021,06/30/2024


In [14]:
# Data types and null counts
print('Data types & null counts:')
info = pd.DataFrame({
    'dtype'     : df_raw.dtypes,
    'null_count': df_raw.isnull().sum(),
    'null_pct'  : (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})
print(info)

Data types & null counts:
                              dtype  null_count  null_pct
Facility Name                object           0    0.0000
Facility ID                   int64           0    0.0000
State                        object           0    0.0000
Measure Name                 object           0    0.0000
Number of Discharges        float64       10088   55.0000
Footnote                    float64       11343   61.9000
Excess Readmission Ratio    float64        6610   36.1000
Predicted Readmission Rate  float64        6610   36.1000
Expected Readmission Rate   float64        6610   36.1000
Number of Readmissions       object        6610   36.1000
Start Date                   object           0    0.0000
End Date                     object           0    0.0000


In [16]:
# CMS suppresses data for small hospitals — footnote column signals this
print('Footnote value counts (suppression flag):')
print(df_raw['Footnote'].value_counts(dropna=False).head(10))

Footnote value counts (suppression flag):
Footnote
NaN        11343
5.0000      3255
1.0000      3150
29.0000      377
7.0000       205
Name: count, dtype: int64


In [18]:
# Distinct measure names — confirmed FY2026 condition codes
print('Distinct measure names (condition codes):')
for m in sorted(df_raw['Measure Name'].dropna().unique()):
    print(f'  {m}')

Distinct measure names (condition codes):
  READM-30-AMI-HRRP
  READM-30-CABG-HRRP
  READM-30-COPD-HRRP
  READM-30-HF-HRRP
  READM-30-HIP-KNEE-HRRP
  READM-30-PN-HRRP


## 3. Clean & Transform

In [23]:
df = df_raw.copy()

# ── 3A. Rename columns for easier handling
df.rename(columns={
    'Facility Name'           : 'facility_name',
    'Facility ID'             : 'facility_id',
    'State'                   : 'state',
    'Measure Name'            : 'condition_code',
    'Number of Discharges'    : 'num_discharges',
    'Footnote'                : 'footnote',
    'Excess Readmission Ratio': 'excess_readmit_ratio',
    'Predicted Readmission Rate': 'predicted_readmit_rate',
    'Expected Readmission Rate' : 'expected_readmit_rate',
    'Number of Readmissions'  : 'num_readmissions',
    'Start Date'              : 'start_date',
    'End Date'                : 'end_date'
}, inplace=True)

# ── 3B. Map condition codes to readable labels
# Exact codes confirmed from FY2026 data
condition_map = {
    'READM-30-AMI-HRRP'      : 'Acute MI',
    'READM-30-CABG-HRRP'     : 'CABG',
    'READM-30-COPD-HRRP'     : 'COPD',
    'READM-30-HF-HRRP'       : 'Heart Failure',
    'READM-30-HIP-KNEE-HRRP' : 'Hip/Knee Arthroplasty',
    'READM-30-PN-HRRP'       : 'Pneumonia'
}
df['condition'] = df['condition_code'].map(condition_map).fillna('Other')

# ── 3C. Numeric type casting
# CMS stores integers as floats (104.0) — convert via float → int
for col in ['num_discharges', 'num_readmissions']:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # float first
    df[col] = df[col].where(df[col].isna(), df[col].astype('Int64'))  # nullable int

for col in ['excess_readmit_ratio', 'predicted_readmit_rate', 'expected_readmit_rate']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── 3D. Date columns
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date']   = pd.to_datetime(df['end_date'],   errors='coerce')

# ── 3E. Derived flags
# Data suppression: CMS hides data for hospitals with too few cases
df['data_suppressed'] = df['footnote'].astype(str).str.strip().isin(['nan', '', 'None']) == False

# Penalty flag: ERR > 1.0 → hospital receives Medicare payment reduction
df['penalty_flag'] = df['excess_readmit_ratio'] > 1.0

# ── 3F. Excess readmissions calculation
# NOTE: expected_readmit_rate is stored as a PERCENTAGE (e.g. 16.93 = 16.93%)
# Must divide by 100 before multiplying by discharges
df['expected_readmissions'] = (
    (df['expected_readmit_rate'] / 100.0) * df['num_discharges']
)
df['excess_readmissions'] = (
    df['num_readmissions'] - df['expected_readmissions']
)
# Positive-only: floor at 0 for cost calculations
df['excess_readmissions_pos'] = df['excess_readmissions'].clip(lower=0)

# Cost estimate @ $15,200 per excess readmission (CMS Medicare average)
df['est_excess_cost_usd'] = df['excess_readmissions_pos'] * 15_200

print(f'Cleaned dataset: {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

Cleaned dataset: 18,330 rows
Columns: ['facility_name', 'facility_id', 'state', 'condition_code', 'num_discharges', 'footnote', 'excess_readmit_ratio', 'predicted_readmit_rate', 'expected_readmit_rate', 'num_readmissions', 'start_date', 'end_date', 'condition', 'data_suppressed', 'penalty_flag', 'expected_readmissions', 'excess_readmissions', 'excess_readmissions_pos', 'est_excess_cost_usd']


## 4. Merge Hospital Dimension Data

In [28]:
# Keep only the hospital attributes we need
hosp_cols = [
    'Facility ID', 'Hospital Type', 'Hospital Ownership',
    'City/Town', 'County/Parish', 'ZIP Code'
]
df_hosp_slim = df_hosp[hosp_cols].copy()
df_hosp_slim.rename(columns={
    'Facility ID'       : 'facility_id',
    'Hospital Type'     : 'hospital_type',
    'Hospital Ownership': 'hospital_ownership',
    'City/Town'         : 'city',
    'County/Parish'     : 'county',
    'ZIP Code'          : 'zip_code'
}, inplace=True)

# ── FIX: convert both join keys to string before merging
df['facility_id']           = df['facility_id'].astype(str).str.strip()
df_hosp_slim['facility_id'] = df_hosp_slim['facility_id'].astype(str).str.strip()

df_merged = df.merge(df_hosp_slim, on='facility_id', how='left')

def ownership_group(val):
    if pd.isna(val):                                    return 'Other / Unknown'
    v = str(val).lower()
    if 'government' in v:                               return 'Government'
    if 'non-profit' in v or 'voluntary' in v:           return 'Non-Profit'
    if 'proprietary' in v or 'for profit' in v:         return 'For-Profit'
    return 'Other / Unknown'

df_merged['ownership_group'] = df_merged['hospital_ownership'].apply(ownership_group)

matched = df_merged['hospital_type'].notna().sum()
print(f'Matched to hospital info: {matched:,} / {len(df_merged):,} '
      f'({matched/len(df_merged)*100:.1f}%)')
print(f'\nOwnership groups:\n{df_merged["ownership_group"].value_counts()}')

Matched to hospital info: 14,940 / 18,330 (81.5%)

Ownership groups:
ownership_group
Non-Profit         9588
Other / Unknown    3804
For-Profit         2898
Government         2040
Name: count, dtype: int64


## 5. Working Datasets

In [31]:
# Full dataset (all 18,330 rows)
df_all = df_merged.copy()

# Complete records only — exclude CMS-suppressed rows (used for all analysis)
df_clean = df_merged[~df_merged['data_suppressed']].copy()

print(f'Total records    : {len(df_all):,}')
print(f'Complete records : {len(df_clean):,}')
print(f'Suppressed       : {len(df_all) - len(df_clean):,}')
print(f'\nConditions       : {df_clean["condition"].nunique()}')
print(f'States           : {df_clean["state"].nunique()}')
print(f'Unique hospitals : {df_clean["facility_id"].nunique():,}')

Total records    : 18,330
Complete records : 11,343
Suppressed       : 6,987

Conditions       : 6
States           : 51
Unique hospitals : 2,748


In [33]:
# Save processed file for notebooks 2 and 3
import os
os.makedirs('data/processed', exist_ok=True)

df_clean.to_csv('data/processed/readmission_clean.csv', index=False)
df_all.to_csv('data/processed/readmission_full.csv', index=False)

print('Saved:')
print('  data/processed/readmission_clean.csv  (complete records only)')
print('  data/processed/readmission_full.csv   (all records including suppressed)')

Saved:
  data/processed/readmission_clean.csv  (complete records only)
  data/processed/readmission_full.csv   (all records including suppressed)


## 6. Quick EDA Summary

In [36]:
print('=' * 55)
print('DATASET SUMMARY — CMS HRRP FY2026')
print('=' * 55)
print(f'  Hospitals analyzed    : {df_clean["facility_id"].nunique():,}')
print(f'  States covered        : {df_clean["state"].nunique()}')
print(f'  Conditions tracked    : {df_clean["condition"].nunique()}')
print(f'  Total discharges      : {df_clean["num_discharges"].sum():,.0f}')
print(f'  Total readmissions    : {df_clean["num_readmissions"].sum():,.0f}')
print(f'  National avg ERR      : {df_clean["excess_readmit_ratio"].mean():.4f}')
print(f'  ERR range             : {df_clean["excess_readmit_ratio"].min():.4f} – '
      f'{df_clean["excess_readmit_ratio"].max():.4f}')
penalized = df_clean[df_clean['penalty_flag']]['facility_id'].nunique()
total     = df_clean['facility_id'].nunique()
print(f'  Penalized hospitals   : {penalized:,} / {total:,} '
      f'({penalized/total*100:.1f}%)')
print(f'  Total excess cost est : ${df_clean["est_excess_cost_usd"].sum()/1e6:.1f}M')
print('=' * 55)

DATASET SUMMARY — CMS HRRP FY2026
  Hospitals analyzed    : 2,748
  States covered        : 51
  Conditions tracked    : 6
  Total discharges      : 2,276,277
  Total readmissions    : 381,794
  National avg ERR      : 1.0020
  ERR range             : 0.4698 – 1.6297
  Penalized hospitals   : 2,284 / 2,748 (83.1%)
  Total excess cost est : $402.2M
